1 — Transaction Parser

In [5]:
df = raw_df.copy()

# Parse mixed date formats.
df["date"] = pd.to_datetime(df["Date"], errors="coerce", dayfirst=True)

# Parse HH:MM time safely.
df["time_clean"] = df["Time"].astype(str).str.strip()
df["hour"] = pd.to_numeric(df["time_clean"].str[:2], errors="coerce")

# Parse mixed currency formats.
df["amount"] = (
    df["Amount"].astype(str)
      .str.replace("₹", "", regex=False)
      .str.replace("Rs.", "", regex=False)
      .str.replace(",", "", regex=False)
      .str.strip()
)
df["amount"] = pd.to_numeric(df["amount"], errors="coerce")

# Standardize transaction type.
type_map = {
    "DR": "debit",
    "Debit": "debit",
    "CR": "credit",
    "Credit": "credit"
}
df["type_clean"] = df["Type"].map(type_map)

# Treat empty modes as missing.
df["Mode"] = df["Mode"].replace(r"^\s*$", np.nan, regex=True)

before_duplicates = len(df)
df = df.drop_duplicates().copy()
duplicates_dropped = before_duplicates - len(df)

bad_dates = int(df["date"].isna().sum())
bad_amounts = int(df["amount"].isna().sum())
bad_types = int(df["type_clean"].isna().sum())

df = df.dropna(subset=["date", "amount", "type_clean"]).copy()
df["month"] = df["date"].dt.month
df["month_name"] = df["date"].dt.strftime("%b")
df["day_of_week"] = df["date"].dt.day_name()

print("=" * 72)
print("SPENDDNA — TRANSACTION PARSER")
print("=" * 72)
print(f"Parsed transactions : {len(df):,}")
print(f"Dropped duplicates   : {duplicates_dropped:,}")
print(f"Unparseable dates    : {bad_dates}")
print(f"Unparseable amounts  : {bad_amounts}")
print(f"Unparseable types    : {bad_types}")
print(f"Date range           : {df['date'].min():%d %B %Y} to {df['date'].max():%d %B %Y}")



SPENDDNA — TRANSACTION PARSER
Parsed transactions : 143
Dropped duplicates   : 18
Unparseable dates    : 1167
Unparseable amounts  : 0
Unparseable types    : 0
Date range           : 01 January 2024 to 06 December 2024


2 — Vendor Extractor

In [6]:
VENDOR_KEYWORDS = {
    "Swiggy": ["SWIGGY", "BUNDL"],
    "Zomato": ["ZOMATO"],
    "Zepto": ["ZEPTO"],
    "Blinkit": ["BLINKIT"],
    "Amazon": ["AMAZON", "AMZN"],
    "Flipkart": ["FLIPKART", "FKART"],
    "Myntra": ["MYNTRA"],
    "Nykaa": ["NYKAA", "FSN E-COMMERCE"],
    "Uber": ["UBER"],
    "Ola": ["OLA"],
    "Rapido": ["RAPIDO"],
    "Starbucks": ["STARBUCKS"],
    "Third Wave Coffee": ["THIRDWAVE", "THIRD WAVE"],
    "Cafe Coffee Day": ["COFFEE DAY", "CCD"],
    "Truffles": ["TRUFFLES"],
    "Restaurant": ["RESTAURANT", "DINEOUT", "MEGHANA", "EMPIRE"],
    "DMart": ["DMART", "AVENUE SUPERMARTS"],
    "BigBasket": ["BIGBASKET"],
    "Grofers": ["GROFERS"],
    "Zerodha": ["ZERODHA"],
    "Groww": ["GROWW"],
    "Netflix": ["NETFLIX"],
    "Spotify": ["SPOTIFY"],
    "Disney+ Hotstar": ["HOTSTAR", "DISNEY"],
    "Jio": ["JIO"],
    "Airtel": ["AIRTEL"],
    "Vi": ["VI POSTPAID", "VI-"],
    "BESCOM": ["BESCOM", "BANGALORE ELEC SUPPLY"],
    "BWSSB": ["BWSSB"],
    "BMTC": ["BMTC", "TUMMOC"],
    "BookMyShow": ["BOOKMYSHOW", "BMS MOVIE"],
    "HP Petrol": ["HP PETROL"],
    "Indian Oil": ["INDIAN OIL", "IOC"],
    "BPCL": ["BPCL"],
    "Rent": ["RENT-LANDLORD", "IMPS-RENT"],
    "Salary": ["SALARY"],
    "P2P Transfer": ["UPI-AMAN", "UPI-ANKIT", "UPI-PRIYA", "UPI-VIKAS", "UPI-NEHA", "UPI-SNEHA", "UPI-KARAN"],
    "Cash Withdrawal": ["ATM-WDL"]
}

def extract_vendor(description):
    text = str(description).upper()
    for vendor, keywords in VENDOR_KEYWORDS.items():
        for keyword in keywords:
            if keyword.upper() in text:
                return vendor
    return "Uncategorised"

df["vendor_clean"] = df["Description"].apply(extract_vendor)

print("Unique canonical vendors:", df["vendor_clean"].nunique())
print("\nTop vendors by transaction count:")
print(df["vendor_clean"].value_counts().head(10))



Unique canonical vendors: 30

Top vendors by transaction count:
vendor_clean
Swiggy           28
Zomato           16
Uncategorised    15
Ola               8
Uber              7
Restaurant        7
Zepto             5
P2P Transfer      4
DMart             4
Starbucks         4
Name: count, dtype: int64


3 — Category Tagger

In [7]:
VENDOR_CATEGORY = {
    "Swiggy": "Food Delivery",
    "Zomato": "Food Delivery",
    "Zepto": "Quick Commerce",
    "Blinkit": "Quick Commerce",
    "Amazon": "Ecommerce",
    "Flipkart": "Ecommerce",
    "Myntra": "Ecommerce",
    "Nykaa": "Ecommerce",
    "Uber": "Transport",
    "Ola": "Transport",
    "Rapido": "Transport",
    "Starbucks": "Cafe",
    "Third Wave Coffee": "Cafe",
    "Cafe Coffee Day": "Cafe",
    "Truffles": "Restaurants",
    "Restaurant": "Restaurants",
    "DMart": "Groceries",
    "BigBasket": "Groceries",
    "Grofers": "Groceries",
    "Zerodha": "Investments",
    "Groww": "Investments",
    "Netflix": "Subscriptions",
    "Spotify": "Subscriptions",
    "Disney+ Hotstar": "Subscriptions",
    "Jio": "Utilities",
    "Airtel": "Utilities",
    "Vi": "Utilities",
    "BESCOM": "Utilities",
    "BWSSB": "Utilities",
    "BMTC": "Transport",
    "BookMyShow": "Entertainment",
    "HP Petrol": "Fuel",
    "Indian Oil": "Fuel",
    "BPCL": "Fuel",
    "Rent": "Utilities",
    "Salary": "Income",
    "P2P Transfer": "Personal Transfer",
    "Cash Withdrawal": "Cash Withdrawal"
}

df["category"] = df["vendor_clean"].map(VENDOR_CATEGORY).fillna("Uncategorised")

print("=" * 72)
print("CATEGORY BREAKDOWN")
print("=" * 72)
print(df["category"].value_counts())
print("\nUncategorised rows:", int((df["category"] == "Uncategorised").sum()))



CATEGORY BREAKDOWN
category
Food Delivery        44
Transport            17
Uncategorised        15
Ecommerce            13
Cafe                  9
Quick Commerce        7
Restaurants           7
Groceries             5
Investments           5
Utilities             5
Personal Transfer     4
Entertainment         4
Cash Withdrawal       3
Income                2
Subscriptions         2
Fuel                  1
Name: count, dtype: int64

Uncategorised rows: 15


4 — Spending Overview

In [8]:
debits = df[df["type_clean"] == "debit"].copy()
credits = df[df["type_clean"] == "credit"].copy()

total_credits = credits["amount"].sum()
total_debits = debits["amount"].sum()
net_savings = total_credits - total_debits
savings_rate = (net_savings / total_credits * 100) if total_credits else np.nan

category_spend = debits.groupby("category")["amount"].sum().sort_values(ascending=False)
vendor_spend = debits.groupby("vendor_clean")["amount"].sum().sort_values(ascending=False)

print("=" * 72)
print("SPENDDNA REPORT — SPENDING OVERVIEW")
print("=" * 72)
print(f"Total credits      : ₹{total_credits:,.2f}")
print(f"Total debits       : ₹{total_debits:,.2f}")
print(f"Net savings        : ₹{net_savings:,.2f}")
print(f"Savings rate       : {savings_rate:.1f}%")
print(f"Transactions       : {len(df):,}")
print(f"Debit transactions : {len(debits):,}")
print("\nTOP 5 CATEGORIES BY SPEND")
for cat, value in category_spend.head(5).items():
    pct = value / total_debits * 100 if total_debits else 0
    print(f"{cat:<22} ₹{value:>10,.2f} ({pct:5.1f}%)")

print("\nTOP 5 VENDORS BY SPEND")
for vendor, value in vendor_spend.head(5).items():
    print(f"{vendor:<22} ₹{value:>10,.2f}")



SPENDDNA REPORT — SPENDING OVERVIEW
Total credits      : ₹170,094.00
Total debits       : ₹205,310.00
Net savings        : ₹-35,216.00
Savings rate       : -20.7%
Transactions       : 143
Debit transactions : 141

TOP 5 CATEGORIES BY SPEND
Investments            ₹ 64,496.00 ( 31.4%)
Utilities              ₹ 40,401.00 ( 19.7%)
Ecommerce              ₹ 36,416.00 ( 17.7%)
Food Delivery          ₹ 19,617.00 (  9.6%)
Groceries              ₹  7,672.00 (  3.7%)

TOP 5 VENDORS BY SPEND
Zerodha                ₹ 60,000.00
Rent                   ₹ 36,000.00
Myntra                 ₹ 13,629.00
Swiggy                 ₹ 12,638.00
Flipkart               ₹ 10,041.00


In [20]:
 #5 — Monthly Trend Analysis

In [21]:
debit_month = (
    debits.pivot_table(
        values="amount",
        index="category",
        columns="month_name",
        aggfunc="sum",
        fill_value=0
    )
)

month_order = ["Jan", "Feb", "Mar", "Apr", "May", "Jun"]
debit_month = debit_month.reindex(columns=[m for m in month_order if m in debit_month.columns], fill_value=0)

print("=" * 72)
print("MONTHLY SPENDING MATRIX (₹)")
print("=" * 72)
print(debit_month.round(2).to_string())

first_col = debit_month.iloc[:, 0]
last_col = debit_month.iloc[:, -1]
growth = pd.Series(np.nan, index=debit_month.index)
mask = first_col > 0
growth.loc[mask] = ((last_col[mask] - first_col[mask]) / first_col[mask]) * 100

print("\nFIRST → LAST MONTH CHANGE")
for cat, value in growth.sort_values(ascending=False).items():
    if pd.notna(value):
        print(f"{cat:<22} {value:>8.1f}%")



MONTHLY SPENDING MATRIX (₹)
month_name            Jan     Feb      Mar     Apr      May     Jun
category                                                           
Cafe                295.0   355.0      0.0   688.0    166.0     0.0
Cash Withdrawal       0.0     0.0      0.0     0.0      0.0     0.0
Ecommerce          6972.0   996.0   2147.0  3311.0      0.0  4868.0
Entertainment         0.0     0.0      0.0   343.0      0.0     0.0
Food Delivery      1180.0  3454.0   2862.0   632.0   1159.0  1636.0
Fuel                  0.0     0.0      0.0     0.0      0.0     0.0
Groceries             0.0     0.0   1213.0     0.0      0.0     0.0
Investments           0.0     0.0      0.0     0.0      0.0     0.0
Personal Transfer  1828.0     0.0      0.0     0.0      0.0  1860.0
Quick Commerce      825.0   697.0      0.0     0.0    502.0     0.0
Restaurants        1794.0     0.0   1748.0     0.0    753.0     0.0
Subscriptions         0.0     0.0      0.0     0.0      0.0     0.0
Transport           

6 — Time-of-Day Patterns

In [11]:
time_matrix = (
    debits.groupby(["category", "hour"])["amount"]
          .sum()
          .unstack(fill_value=0)
          .reindex(columns=range(24), fill_value=0)
)

print("=" * 72)
print("TIME-OF-DAY SPENDING HEATMAP (₹)")
print("=" * 72)
print("       " + " ".join(f"{h:02d}" for h in range(24)))

def intensity_symbol(value, row_max):
    if value == 0 or row_max == 0:
        return "."
    ratio = value / row_max
    if ratio <= 0.25:
        return "░"
    if ratio <= 0.50:
        return "▒"
    if ratio <= 0.75:
        return "▓"
    return "█"

for category, row in time_matrix.iterrows():
    row_max = row.max()
    symbols = " ".join(intensity_symbol(v, row_max) for v in row)
    print(f"{category[:12]:<12} {symbols}")

# Late-night food delivery insight.
food_delivery = debits[debits["category"] == "Food Delivery"]
if len(food_delivery):
    late = food_delivery[food_delivery["hour"].isin([21,22,23,0,1])]
    late_pct = len(late) / len(food_delivery) * 100
    print(f"\nLate-night Food Delivery transactions (21:00–01:59): {late_pct:.1f}%")



TIME-OF-DAY SPENDING HEATMAP (₹)
       00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23
Cafe         . . . . . . . . ░ ▒ █ . . . ░ . ▒ ░ . . . . . .
Cash Withdra . . . . . . . . . ░ █ . . . ░ . . . . . . . . .
Ecommerce    . ░ . ░ . . ░ ░ . . . ▒ . ░ . ░ . . ▓ ▒ █ ▒ . .
Entertainmen ▓ ▓ . . . . . . . . . . . . . . █ . . . . . ▒ .
Food Deliver ░ . ░ . ░ ░ . . ░ ░ . █ █ ▒ ░ ▒ ░ ▒ ▓ ▒ ▓ ▓ ▓ ▒
Fuel         . . . . . . . . . . . . . . . . . . . █ . . . .
Groceries    . . . . . . . . █ . . █ . . . . . . ▒ . ▒ . . .
Investments  . . . . ▒ ░ . . . . █ . . . . . . . . . . . . .
Personal Tra . . . . . . . . . . . . . . █ . ░ . . . . ▓ █ .
Quick Commer . . ▓ . . . . . ▒ . . . . . █ . . ░ ▒ . . . . .
Restaurants  . . . . . . . . . . ▒ ░ . █ . . . . ▓ . █ . . .
Subscription . . . . . . . . . . . . ░ . █ . . . . . . . . .
Transport    ▒ . . . . ▒ ▒ ▒ . ▒ . . . ░ ░ █ ▓ . ▒ . ▒ █ . .
Uncategorise ▒ . . ▓ . . . ▒ . █ ▓ . . ▒ . . ▒ ▒ ░ . ▒ . . ▓
Utilities    . . . . . . . . . . .

 7 — Anomaly Detection

In [12]:
category_mean = df.groupby("category")["amount"].transform("mean")
category_std = df.groupby("category")["amount"].transform("std")

df["z_score"] = np.where(
    category_std > 0,
    (df["amount"] - category_mean) / category_std,
    0
)

anomalies = (
    df[(df["type_clean"] == "debit") & (df["z_score"] > 2)]
    .sort_values("z_score", ascending=False)
)

print("=" * 72)
print("ANOMALY DETECTION — TOP TRANSACTIONS")
print("=" * 72)
if anomalies.empty:
    print("No transactions exceeded z-score > 2.")
else:
    print(anomalies[["date", "vendor_clean", "category", "amount", "z_score"]]
          .head(10)
          .to_string(index=False, formatters={"amount": "₹{:,.2f}".format,
                                               "z_score": "{:.2f}".format}))
print(f"\nTotal anomalies (z > 2): {len(anomalies)}")



ANOMALY DETECTION — TOP TRANSACTIONS
      date  vendor_clean      category     amount z_score
2024-09-01        Myntra     Ecommerce ₹10,745.00    3.07
2024-01-02 Uncategorised Uncategorised    ₹791.00    2.03

Total anomalies (z > 2): 2


 8 — Spending Archetype Detection

In [17]:
def pct_of_debits(category):
    value = category_spend.get(category, 0)
    return value / total_debits * 100 if total_debits else 0

foodie_pct = pct_of_debits("Food Delivery") + pct_of_debits("Restaurants") + pct_of_debits("Cafe")
quick_pct = pct_of_debits("Quick Commerce")
ecommerce_pct = pct_of_debits("Ecommerce")
investment_pct = pct_of_debits("Investments")
transport_pct = pct_of_debits("Transport")
subscription_vendors = debits.loc[debits["category"] == "Subscriptions", "vendor_clean"].nunique()
yolo = savings_rate < 10

late_night_food_pct = (
    len(food_delivery[food_delivery["hour"].isin([21,22,23,0,1])]) / len(food_delivery) * 100
    if len(food_delivery) else 0
)

archetypes = []

if foodie_pct > 25:
    archetypes.append(("THE FOODIE", f"Food + Restaurants + Cafe = {foodie_pct:.1f}% of debits"))
if quick_pct > 15:
    archetypes.append(("THE QUICK COMMERCE JUNKIE", f"Quick Commerce = {quick_pct:.1f}% of debits"))
if ecommerce_pct > 15:
    archetypes.append(("THE SHOPAHOLIC", f"Ecommerce = {ecommerce_pct:.1f}% of debits"))
if investment_pct > 15:
    archetypes.append(("THE INVESTOR", f"Investments = {investment_pct:.1f}% of debits"))
if late_night_food_pct > 50:
    archetypes.append(("THE LATE-NIGHT SNACKER", f"{late_night_food_pct:.1f}% of Food Delivery transactions occur 21:00–01:59"))
if transport_pct > 10:
    archetypes.append(("THE CAB COMMUTER", f"Transport = {transport_pct:.1f}% of debits"))
if subscription_vendors >= 5:
    archetypes.append(("THE SUBSCRIPTION LOVER", f"{subscription_vendors} distinct subscription vendors"))
if yolo:
    archetypes.append(("THE YOLO SPENDER", f"Savings rate = {savings_rate:.1f}%"))
if savings_rate > 40:
    archetypes.append(("THE DISCIPLINED SAVER", f"Savings rate = {savings_rate:.1f}%"))

# Bonus ninth archetype: Bengaluru Coffee Regular
cafe_pct = pct_of_debits("Cafe")
if cafe_pct >= 3:
    archetypes.append(("THE BENGALURU COFFEE REGULAR", f"Cafe spend = {cafe_pct:.1f}% of debits"))

print("=" * 72)
print("SPENDING ARCHETYPES")
print("=" * 72)
if archetypes:
    for label, metric in archetypes:
        print(f"{label:<32} | {metric}")
else:
    print("No archetype rules were triggered.")

SPENDING ARCHETYPES
THE SHOPAHOLIC                   | Ecommerce = 17.7% of debits
THE INVESTOR                     | Investments = 31.4% of debits
THE YOLO SPENDER                 | Savings rate = -20.7%


In [ ]:
 9 — Day-of-Week Analysis

In [15]:
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
dow_spend = debits.groupby("day_of_week")["amount"].sum().reindex(weekday_order, fill_value=0)

weekday_total = dow_spend.iloc[:5].sum()
weekend_total = dow_spend.iloc[5:].sum()

print("=" * 72)
print("DAY-OF-WEEK SPENDING")
print("=" * 72)
for day, value in dow_spend.items():
    print(f"{day:<12} ₹{value:>12,.2f}")

print(f"\nWeekday total : ₹{weekday_total:,.2f}")
print(f"Weekend total : ₹{weekend_total:,.2f}")
print(f"Highest day   : {dow_spend.idxmax()} (₹{dow_spend.max():,.2f})")



DAY-OF-WEEK SPENDING
Monday       ₹   17,106.00
Tuesday      ₹   35,776.00
Wednesday    ₹   29,859.00
Thursday     ₹   11,988.00
Friday       ₹   43,159.00
Saturday     ₹    5,260.00
Sunday       ₹   62,162.00

Weekday total : ₹137,888.00
Weekend total : ₹67,422.00
Highest day   : Sunday (₹62,162.00)
